# Integrated acceptance calibration
CPU-only Ada workflow using the canonical precision-aware package.
The original notebook and synthetic bundle remain historical evidence. Run from the full repository checkout.
Synthetic results are for debugging only. Real traces require the shared collector, precision audit and exact manifest.


## 1. Settings

In [ ]:
from pathlib import Path

USE_SYNTHETIC = True            # set False once Lucia's traces arrive

# Where Lucia's files live. In Colab with Drive mounted, for example:
#   from google.colab import drive; drive.mount("/content/drive")
#   DATA_DIR = Path("/content/drive/MyDrive/ECE285_Medusa/20261028_192748_UTC")
DATA_DIR = Path("data")
CALIB_FILE = DATA_DIR / "calib.npz"              # AlpacaEval traces (train + validation prompts)
EVAL_FILE = DATA_DIR / "eval.npz"                # MT-Bench traces (optional for this notebook)
MANIFEST_FILE = DATA_DIR / "data_manifest.json"  # Lucia's fixed train/validation prompt IDs

OUT_DIR = Path("outputs_ada_integrated")
SEED = 0
BUDGETS_FOR_ALEX = [8, 16, 32, 63, 96]   # fixed-tree sizes Alex needs
POOL_BUDGET = 64          # nodes per step in the shared candidate pool (MEDUSA+ trees)
POOL_STEPS = 4000         # steps sampled to build pools (lower = faster)
QUICK_EVAL_STEPS = 800    # steps for quick accepted-token checks on validation

# Restart the kernel before switching from the original notebook.
import sys
REPO = Path.cwd().resolve()
if not (REPO / "medusa-proj-skeleton/src/mcal").is_dir():
    raise RuntimeError("Run this notebook from the repository root")
sys.path.insert(0, str(REPO / "medusa-proj-skeleton/src"))
if any(k == "mcal" or k.startswith("mcal.") for k in sys.modules):
    raise RuntimeError("Restart the kernel to ensure canonical mcal imports")
OUT_DIR = OUT_DIR / ("synthetic" if USE_SYNTHETIC else "real")
if OUT_DIR.exists():
    raise RuntimeError("Choose a new OUT_DIR; previous results are preserved")
OUT_DIR.mkdir(parents=True)
PRECISION_MODE = "nf4_fp32_from_fp16"  # provisional until the GPU correctness gate passes

## Shared implementation
Imports use medusa-proj-skeleton/src/mcal. No source files are overwritten.


Shared module imported from the canonical package; original embedded code remains in the historical notebook.


Shared module imported from the canonical package; original embedded code remains in the historical notebook.


Shared module imported from the canonical package; original embedded code remains in the historical notebook.


Shared module imported from the canonical package; original embedded code remains in the historical notebook.


Shared module imported from the canonical package; original embedded code remains in the historical notebook.


Shared module imported from the canonical package; original embedded code remains in the historical notebook.


## 3. Load and check the traces

In [ ]:
import importlib, json, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mcal.traces, mcal.features, mcal.estimators, mcal.tree, mcal.evaluate
for m in [mcal.traces, mcal.features, mcal.estimators, mcal.tree, mcal.evaluate]:
    importlib.reload(m)   # picks up edits after re-running the %%writefile cells

from mcal.traces import TraceSet, make_synthetic_traces
from mcal.features import (NodeBatch, conditional_nodes, candidate_pool_nodes,
                           node_features, FEATURE_GROUPS)
from mcal.estimators import (Scorer, MedusaAccuracy, MedusaPlusProb, CountConditional,
                             LogisticConditional, DirectClassifier)
from mcal.tree import select_tree, fixed_tree, tree_to_medusa_choices, is_ancestor_closed
from mcal.evaluate import evaluate_tree_policy, brier, ece, reliability

pd.set_option("display.precision", 4)

In [ ]:
if USE_SYNTHETIC:
    calib = make_synthetic_traces(n_prompts=250, steps_per_prompt=120, seed=SEED)
    evals = make_synthetic_traces(n_prompts=80, steps_per_prompt=120, seed=SEED + 100)
else:
    calib = TraceSet.load(CALIB_FILE)
    evals = TraceSet.load(EVAL_FILE) if EVAL_FILE.exists() else None

print(calib.summary())
print(evals.summary() if evals is not None else "No eval traces yet: eval-set calibration is skipped.")
print("metadata:", calib.meta)
from mcal.precision import trace_contract
from mcal.features import FEATURE_SCHEMA
from mcal.audit import validate_arrays, audit_file, audit_pair
from mcal.config import load_config
validate_arrays(calib)
if evals is not None:
    validate_arrays(evals)
if not USE_SYNTHETIC:
    audit_cfg = load_config(str(REPO / "medusa-proj-skeleton/configs/default.yaml"),
                            ["model.precision_mode=" + PRECISION_MODE])
    audit_file(CALIB_FILE, audit_cfg)
    if evals is not None:
        audit_file(EVAL_FILE, audit_cfg)
        audit_pair(calib, evals)
contract = trace_contract(calib)


In [ ]:
def check_traces(ts, name):
    """Catch format problems in Lucia's export before they turn into strange results."""
    H, K = ts.n_heads, ts.topk
    problems = []
    if ts.topk_prob.shape != (ts.n_steps, H, K):
        problems.append(f"topk_prob shape {ts.topk_prob.shape}, expected {(ts.n_steps, H, K)}")
    if ts.gold_rank.min() < -1 or ts.gold_rank.max() >= K:
        problems.append("gold_rank outside [-1, K-1]")
    p = ts.topk_prob
    if (p < 0).any() or (p.sum(-1) > 1 + 1e-3).any():
        problems.append("head probabilities negative or summing above 1")
    if (np.diff(p, axis=-1) > 1e-6).any():
        problems.append("top-K probabilities not sorted high to low")
    if not ((ts.root_prob > 0) & (ts.root_prob <= 1 + 1e-6)).all():
        problems.append("root_prob outside (0, 1]")
    if K != 10:
        problems.append(f"K = {K}; Medusa's TOPK is 10")
    if len(np.unique(ts.prompt_id)) < 2:
        problems.append("fewer than 2 prompts: a prompt-level split is impossible")
    print(f"{name}: " + ("format OK" if not problems else "PROBLEMS: " + "; ".join(problems)))

    ranks = pd.DataFrame(
        {f"head {j + 1}": np.bincount(ts.gold_rank[:, j] + 1, minlength=K + 1) / ts.n_steps
         for j in range(H)},
        index=["not in top-10"] + [f"rank {r}" for r in range(K)]).T
    display(ranks.round(3))
    # If ranks 5-9 are nearly empty for deep heads, the export may be limited to the
    # default 63-node tree instead of all top-10 ranks; ask Lucia.

check_traces(calib, "calib")
if evals is not None:
    check_traces(evals, "eval")

## 4. Train/validation split

Always split by prompt (steps within one prompt are correlated). With real data, use Lucia's fixed 200/50 split
so every result in the report uses the same held-out prompts. The pilot (20 training prompts) has no
validation prompts yet; it falls back to a random 80/20 prompt split until the full set arrives.

In [ ]:
from mcal.handoff import split_calibration
def split_train_val(calib):
    manifest = None if USE_SYNTHETIC else MANIFEST_FILE
    train, val = split_calibration(calib, manifest, .8, SEED)
    return train, val, "synthetic prompt split" if USE_SYNTHETIC else "exact manifest"
    
train, val, split_how = split_train_val(calib)
print(f"Split: {split_how}")
print(f"  train: {train.n_steps} steps, {len(np.unique(train.prompt_id))} prompts")
print(f"  val:   {val.n_steps} steps, {len(np.unique(val.prompt_id))} prompts")

t0 = time.time()
train_cond = conditional_nodes(train)   # children of accepted parents, labels S_v | S_u = 1
val_cond = conditional_nodes(val)
print(f"conditional training nodes: train {len(train_cond[0])}, val {len(val_cond[0])} "
      f"({time.time() - t0:.1f}s)")

## 5. Sanity checks

With independent heads, P(head j correct | earlier heads correct) equals the marginal accuracy, so the
count-based conditional must match the product. With dependent heads it must be higher.
If either fails, it's a bug, not a finding. The last table shows the same comparison on the training data.

In [ ]:
rows = []
for dep in [0.0, 1.0]:
    t = make_synthetic_traces(200, 100, dependence=dep, seed=1)
    acc = MedusaAccuracy().fit(t)
    cnt = CountConditional(alpha=1e-9).fit(t)      # no smoothing = raw empirical rates
    for j in range(1, t.n_heads):
        rows.append(dict(dependence=dep, head=j + 1, marginal_top1=acc.acc[j, 0],
                         cond_given_parent_top1=cnt.q[j, 0, 0]))
sanity = pd.DataFrame(rows)
sanity["gap"] = sanity.cond_given_parent_top1 - sanity.marginal_top1
display(sanity.round(3))

indep = sanity[sanity.dependence == 0]
dep = sanity[sanity.dependence == 1]
assert indep.gap.abs().max() < 0.03, "independent heads: conditional should match marginal"
assert (dep.gap > 0.03).all(), "dependent heads: conditional should exceed marginal"
print("Sanity checks passed.")

acc_tr = MedusaAccuracy().fit(train)
cnt_tr = CountConditional(alpha=1e-9).fit(train)
display(pd.DataFrame([dict(head=j + 1, marginal_top1=acc_tr.acc[j, 0],
                           cond_given_parent_top1=cnt_tr.q[j, 0, 0],
                           n_parent_cases=int(cnt_tr.counts[j, 0, 0]))
                      for j in range(1, train.n_heads)]).round(3))

## 6. Tuning on the validation prompts

**Count table `alpha`.** Smoothing shrinks each cell (head, parent rank, rank) toward the same cell pooled
over parent ranks. Very large `alpha` = no parent-rank context at all (the "conditional on acceptance only"
ablation), so the right end of this curve is that ablation for free.

**Logistic `C` and `rank_cap`.** The direct classifier reuses the winning values, so both learned models
keep the same features and capacity.

Scored by Brier score on the validation conditional nodes, which is exactly what the edge factor
q = P(S_v | S_u = 1) is supposed to predict.

In [ ]:
def log_loss(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

alpha_rows = []
for a in [0.1, 1, 3, 10, 30, 100, 300, 1000, 1e9]:
    sc = CountConditional(alpha=a).fit(train, nodes=train_cond)
    p = sc.factor(val, val_cond[0])
    alpha_rows.append(dict(alpha=a, brier=brier(p, val_cond[1]), log_loss=log_loss(p, val_cond[1])))
alpha_df = pd.DataFrame(alpha_rows)
BEST_ALPHA = float(alpha_df.loc[alpha_df.brier.idxmin(), "alpha"])
display(alpha_df)
print("BEST_ALPHA =", BEST_ALPHA)

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.semilogx(alpha_df.alpha[:-1], alpha_df.brier[:-1], "o-", label="count table")
ax.axhline(alpha_df.brier.iloc[-1], ls="--", c="gray", label="no parent-rank context")
ax.set_xlabel("alpha (smoothing)"); ax.set_ylabel("validation Brier"); ax.legend()
fig.tight_layout(); fig.savefig(OUT_DIR / "alpha_tuning.png", dpi=150); plt.show()

In [ ]:
t0 = time.time()
grid_rows = []
for rank_cap in [4, 9]:
    for C in [0.1, 1.0, 10.0]:
        sc = LogisticConditional(C=C, rank_cap=rank_cap).fit(train, nodes=train_cond)
        p = sc.factor(val, val_cond[0])
        grid_rows.append(dict(rank_cap=rank_cap, C=C, brier=brier(p, val_cond[1]),
                              log_loss=log_loss(p, val_cond[1])))
grid_df = pd.DataFrame(grid_rows).sort_values("brier")
best = grid_df.iloc[0]
BEST_C, BEST_RANK_CAP = float(best.C), int(best.rank_cap)
display(grid_df)
print(f"BEST_C = {BEST_C}, BEST_RANK_CAP = {BEST_RANK_CAP}  ({time.time() - t0:.0f}s)")

## 7. Fit all five scorers

The direct classifier needs a candidate pool: the nodes of a MEDUSA+ tree of `POOL_BUDGET` nodes at
sampled steps, labeled with full-prefix acceptance S_v (parents may have failed). Section 9 tests
whether a different pool makes it a stronger baseline.

In [ ]:
t0 = time.time()
ref = MedusaPlusProb()
pool_train = candidate_pool_nodes(train, ref, POOL_BUDGET, POOL_STEPS, seed=SEED)
print(f"direct-classifier pool: {len(pool_train[0])} nodes, "
      f"acceptance rate {pool_train[1].mean():.3f} ({time.time() - t0:.0f}s)")

scorers = {
    "medusa_acc": MedusaAccuracy().fit(train),
    "medusa_plus": ref,
    "count_cond": CountConditional(alpha=BEST_ALPHA).fit(train, nodes=train_cond),
    "logistic_cond": LogisticConditional(C=BEST_C, rank_cap=BEST_RANK_CAP).fit(train, nodes=train_cond),
    "direct": DirectClassifier(C=BEST_C, rank_cap=BEST_RANK_CAP).fit(train, pool=pool_train),
}
print("fitted:", list(scorers), f"({time.time() - t0:.0f}s total)")

## 8. Calibration

* **Edge level** (validation conditional nodes): does each factor predict P(S_v | parent accepted)?
  Marginal-product factors are not meant to, and that gap is the project's claim. The direct classifier
  has no edge factor, so it appears only at path level.
* **Path level** (shared MEDUSA+ candidate pool): does each scorer's path score predict P(S_v)?
  Every method is scored on the same nodes.

In [ ]:
edge_rows, edge_curves = [], {}
for name in ["medusa_acc", "medusa_plus", "count_cond", "logistic_cond"]:
    p = np.clip(scorers[name].factor(val, val_cond[0]), 0, 1)
    y = val_cond[1]
    edge_rows.append(dict(method=name, brier=brier(p, y), ece=ece(p, y), log_loss=log_loss(p, y),
                          mean_pred=p.mean(), mean_obs=y.mean()))
    edge_curves[name] = reliability(p, y.astype(float))
edge_df = pd.DataFrame(edge_rows)
print("Edge level: P(S_v | parent accepted) on validation prompts")
display(edge_df)


def path_calibration(ts, split_name, seed):
    pool = candidate_pool_nodes(ts, ref, POOL_BUDGET, POOL_STEPS, seed=seed)
    rows, curves = [], {}
    for name, sc in scorers.items():
        p = np.clip(sc.path_scores(ts, pool[0]), 0, 1)
        y = pool[1]
        rows.append(dict(split=split_name, method=name, brier=brier(p, y), ece=ece(p, y),
                         mean_pred=p.mean(), mean_obs=y.mean()))
        curves[name] = reliability(p, y.astype(float))
    return rows, curves

t0 = time.time()
path_rows, path_curves = path_calibration(val, "val", seed=1)
if evals is not None:
    eval_rows, path_curves = path_calibration(evals, "synthetic eval" if USE_SYNTHETIC else "eval (MT-Bench)", seed=2)
    path_rows += eval_rows
path_df = pd.DataFrame(path_rows)
print(f"Path level: P(S_v) on a shared MEDUSA+ candidate pool ({time.time() - t0:.0f}s)")
display(path_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, curves, title in [(axes[0], edge_curves, "Edge level, validation"),
                          (axes[1], path_curves, "Path level, " + ("eval" if evals is not None else "validation"))]:
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    for name, (mp, obs, cnt) in curves.items():
        ax.plot(mp, obs, "o-", ms=3, label=name)
    ax.set_xlabel("predicted acceptance"); ax.set_ylabel("observed acceptance")
    ax.set_title(title); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT_DIR / "reliability.png", dpi=150); plt.show()

## 9. Direct-classifier investigation

On synthetic data, the direct classifier trained on a MEDUSA+ pool loses accepted tokens as the budget
grows. The likely cause: it is trained only on nodes MEDUSA+ picks, but at large budgets its own trees
reach nodes it never saw. A weak baseline makes our result look better than it is, so we try three pools
and keep the best for the report:

1. MEDUSA+ trees at `POOL_BUDGET` (current).
2. Larger MEDUSA+ trees (128 nodes).
3. The current pool plus the nodes its own trees pick (one round of retraining, DAgger-style).

Selection uses validation prompts only (never MT-Bench). The `predicted` column is the sum of path scores
over the tree; close to `accepted` means the scorer knows how good its trees are.

In [ ]:
def concat_pools(*pools):
    return NodeBatch.concat([p[0] for p in pools]), np.concatenate([p[1] for p in pools])

t0 = time.time()
mk = lambda: DirectClassifier(C=BEST_C, rank_cap=BEST_RANK_CAP)
pool_big = candidate_pool_nodes(train, ref, 128, POOL_STEPS // 2, seed=SEED)
pool_own = candidate_pool_nodes(train, [ref, scorers["direct"]], 96, POOL_STEPS // 2, seed=SEED + 1)
direct_variants = {
    "direct: MEDUSA+ pool": scorers["direct"],
    "direct: MEDUSA+ pool, 128 nodes": mk().fit(train, pool=pool_big),
    "direct: + own trees (1 round)": mk().fit(train, pool=concat_pools(pool_train, pool_own)),
}
print(f"variants fitted ({time.time() - t0:.0f}s)")

quick = np.sort(np.random.default_rng(SEED).choice(val.n_steps, min(QUICK_EVAL_STEPS, val.n_steps),
                                                   replace=False))
compare = {"medusa_plus": scorers["medusa_plus"], "logistic_cond": scorers["logistic_cond"],
           **direct_variants}
t0 = time.time()
rows = []
for B in [16, 63]:
    for name, sc in compare.items():
        r = evaluate_tree_policy(sc, val, B, steps=quick)
        rows.append(dict(budget=B, method=name, accepted=r["mean_accepted"], sem=r["sem_accepted"],
                         predicted=r["mean_predicted"]))
direct_df = pd.DataFrame(rows)
display(direct_df)
print(f"({time.time() - t0:.0f}s)")

at63 = direct_df[(direct_df.budget == 63) & direct_df.method.str.startswith("direct")]
BEST_DIRECT = at63.loc[at63.accepted.idxmax(), "method"]
scorers["direct"] = direct_variants[BEST_DIRECT]
print("Keeping for the report:", BEST_DIRECT)
# Refresh reported path calibration for the selected final direct model.
path_rows, path_curves = path_calibration(val, "val", seed=1)
if evals is not None:
    eval_rows, path_curves = path_calibration(evals, "synthetic eval" if USE_SYNTHETIC else "eval (MT-Bench)", seed=2)
    path_rows += eval_rows
path_df = pd.DataFrame(path_rows)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, curves, title in [(axes[0], edge_curves, "Edge level, validation"),
                          (axes[1], path_curves, "Path level, " + ("eval" if evals is not None else "validation"))]:
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    for name, (mp, obs, cnt) in curves.items():
        ax.plot(mp, obs, "o-", ms=3, label=name)
    ax.set_xlabel("predicted acceptance"); ax.set_ylabel("observed acceptance")
    ax.set_title(title); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT_DIR / "reliability.png", dpi=150); plt.show()

## 10. Feature-group ablation (logistic conditional)

Each row drops one feature group and refits. A larger `delta_brier` means the group matters more.
`parent_rank` and `interactions` carry the parent-child information the project is about.
Rerun this on the full traces for the report.

In [ ]:
t0 = time.time()
full = LogisticConditional(C=BEST_C, rank_cap=BEST_RANK_CAP).fit(train, nodes=train_cond)
base_brier = brier(full.factor(val, val_cond[0]), val_cond[1])
rows = [dict(dropped="(none)", brier=base_brier, delta_brier=0.0)]
for g in FEATURE_GROUPS:
    sc = LogisticConditional(C=BEST_C, rank_cap=BEST_RANK_CAP, drop=(g,)).fit(train, nodes=train_cond)
    b = brier(sc.factor(val, val_cond[0]), val_cond[1])
    rows.append(dict(dropped=g, brier=b, delta_brier=b - base_brier))
ablation_df = pd.DataFrame(rows).sort_values("delta_brier", ascending=False)
display(ablation_df)
print(f"({time.time() - t0:.0f}s)")

## 11. Save everything for the next step

* `models/<name>.pkl`: the five fitted scorers (load with `Scorer.load`; needs the same `mcal/` folder).
* `models/fixed_tree_<name>_B<budget>.json`: fixed trees in Medusa's `medusa_choices` format.
* `tables/*.csv`, `*.png`: calibration and tuning results for the report.
* `settings.json`: the chosen hyperparameters and which data they came from.
* `models_for_step3.zip`: `models/` plus the `mcal/` code, ready to send.

In [ ]:
# Save the exact audited traces used for this run alongside models.
calib.save(OUT_DIR / "traces/calib.npz")
if evals is not None:
    evals.save(OUT_DIR / "traces/eval.npz")
(OUT_DIR / "split_manifest.json").write_text(json.dumps(dict(
    train_prompt_ids=np.unique(train.prompt_id).tolist(),
    validation_prompt_ids=np.unique(val.prompt_id).tolist()), indent=2))
models_dir, tables_dir = OUT_DIR / "models", OUT_DIR / "tables"
models_dir.mkdir(parents=True, exist_ok=True); tables_dir.mkdir(parents=True, exist_ok=True)

for name, sc in scorers.items():
    sc.execution_contract = contract
    sc.save(models_dir / f"{name}.pkl")

for name in ["medusa_acc", "count_cond"]:            # static scorers -> fixed trees
    for B in BUDGETS_FOR_ALEX:
        tree = fixed_tree(scorers[name], train, B)
        assert len(tree) == B and is_ancestor_closed(tree)
        (models_dir / f"fixed_tree_{name}_B{B}.json").write_text(json.dumps(dict(choices=tree_to_medusa_choices(tree), execution_contract=contract)))

for df, fname in [(alpha_df, "alpha_tuning"), (grid_df, "logistic_tuning"), (edge_df, "calibration_edge"),
                  (path_df, "calibration_path"), (direct_df, "direct_variants"), (ablation_df, "feature_ablation")]:
    df.to_csv(tables_dir / f"{fname}.csv", index=False)

settings = dict(
    execution_contract=contract, feature_schema=FEATURE_SCHEMA,
    data="synthetic" if USE_SYNTHETIC else str(CALIB_FILE), trace_meta=calib.meta, split=split_how,
    train_prompts=int(len(np.unique(train.prompt_id))), val_prompts=int(len(np.unique(val.prompt_id))),
    count_alpha=BEST_ALPHA, logistic_C=BEST_C, rank_cap=BEST_RANK_CAP, direct_pool=BEST_DIRECT,
    pool_budget=POOL_BUDGET, pool_steps=POOL_STEPS, seed=SEED,
)
(OUT_DIR / "settings.json").write_text(json.dumps(settings, indent=2, default=str))

# Round-trip check: the saved files give the same scores when reloaded.
probe = NodeBatch(*[a[:500] for a in (val_cond[0].step, val_cond[0].path, val_cond[0].depth)])
for name, sc in scorers.items():
    again = Scorer.load(models_dir / f"{name}.pkl")
    assert np.allclose(again.path_scores(val, probe), sc.path_scores(val, probe)), name

bundle = OUT_DIR / "bundle"
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(models_dir, bundle / "models")
shutil.copytree(REPO / "medusa-proj-skeleton/src/mcal", bundle / "mcal", ignore=shutil.ignore_patterns("__pycache__"))
shutil.copy(OUT_DIR / "settings.json", bundle / "settings.json")
zip_path = shutil.make_archive(str(OUT_DIR / "models_for_step3"), "zip", bundle)
print("Saved:", sorted(p.name for p in models_dir.iterdir())[:6], "...")
print("Send to Alex:", zip_path)

## Remaining steps
Use a fresh output directory for each run. Do not use synthetic tuned parameters as final research settings.
Real data must pass the CPU audit and representative offline/online GPU correctness validation.
For CLI comparisons, point out_dir to this OUT_DIR and configure the same manifest and trace paths.
Run latency benchmarks separately on the GPU after correctness passes.
